# Kaggle Train Qwen2.5-7B Fixed-Span Viettel

Notebook này train `Qwen/Qwen2.5-7B-Instruct` theo hướng:

- giữ nguyên `span/type/position`
- model chỉ học `assertions` và `candidates`
- clone repo ngay trong notebook

Bạn có 2 cách lấy data:

1. `DATA_SOURCE = "repo"`: đọc `sft_train.jsonl` và `sft_dev.jsonl` trực tiếp từ repo đã push.
2. `DATA_SOURCE = "kaggle_dataset"`: code clone từ repo, nhưng data lấy từ `/kaggle/input/...`.

Khuyến nghị:

- nếu data nhỏ và bạn muốn gọn: để luôn trong repo
- nếu data lớn hơn: để code ở repo, data ở Kaggle Dataset

In [ ]:
REPO_URL = 'https://github.com/QuangVoAI/VTAR.git'
REPO_BRANCH = 'luong-ontology-ai-y-khoa'
REPO_NAME = 'VTAR'

# Nếu repo private, tạo Kaggle secret tên GITHUB_TOKEN rồi bỏ comment đoạn bên dưới.
# import os
# token = os.environ['GITHUB_TOKEN']
# REPO_URL = f'https://{token}@github.com/QuangVoAI/VTAR.git'

!git clone --branch "$REPO_BRANCH" "$REPO_URL" "/kaggle/working/$REPO_NAME"
%cd /kaggle/working/$REPO_NAME
!pip install -q -U transformers datasets peft accelerate bitsandbytes

In [ ]:
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.backends.cuda.matmul.allow_tf32 = True

REPO_DIR = Path(f'/kaggle/working/{REPO_NAME}')

# Chọn 'repo' hoặc 'kaggle_dataset'
DATA_SOURCE = 'repo'

# Nếu DATA_SOURCE='repo', path sẽ trỏ vào repo bạn đã push.
REPO_DATASET_DIR = REPO_DIR / 'artifacts' / 'qwen_fixed_span_68_100'

# Nếu DATA_SOURCE='kaggle_dataset', sửa tên folder input bên dưới.
KAGGLE_DATASET_DIR = Path('/kaggle/input/vtr-qwen-fixed-span')

DATASET_DIR = REPO_DATASET_DIR if DATA_SOURCE == 'repo' else KAGGLE_DATASET_DIR
TRAIN_FILE = DATASET_DIR / 'sft_train.jsonl'
DEV_FILE = DATASET_DIR / 'sft_dev.jsonl'
SUMMARY_FILE = DATASET_DIR / 'summary.json'

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = Path('/kaggle/working/qwen25-7b-fixed-span-lora')

MAX_LENGTH = 1536
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 10
SAVE_STRATEGY = 'epoch'
EVAL_STRATEGY = 'epoch'
SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

assert TRAIN_FILE.exists(), f'Missing {TRAIN_FILE}'
assert DEV_FILE.exists(), f'Missing {DEV_FILE}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('repo dir:', REPO_DIR)
print('dataset dir:', DATASET_DIR)
print('cuda available:', torch.cuda.is_available())
if SUMMARY_FILE.exists():
    print(SUMMARY_FILE.read_text(encoding='utf-8'))

In [ ]:
def load_jsonl(path: Path):
    rows = []
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        raw_line = raw_line.strip()
        if raw_line:
            rows.append(json.loads(raw_line))
    return rows

train_rows = load_jsonl(TRAIN_FILE)
dev_rows = load_jsonl(DEV_FILE)

print('train rows:', len(train_rows))
print('dev rows:', len(dev_rows))
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:2000])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assert tokenizer.chat_template, 'Tokenizer phải có chat_template để format dữ liệu chat.'

def preprocess_row(row):
    encoded = tokenizer.apply_chat_template(
        row['messages'],
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=MAX_LENGTH,
        return_dict=True,
        return_assistant_tokens_mask=True,
    )
    input_ids = list(encoded['input_ids'])
    attention_mask = list(encoded['attention_mask'])
    assistant_mask = list(encoded.get('assistant_tokens_mask', []))
    if not assistant_mask:
        raise ValueError('chat_template không trả assistant token mask; cần tokenizer hỗ trợ generation block.')
    labels = [token_id if mask else -100 for token_id, mask in zip(input_ids, assistant_mask)]
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

train_dataset = Dataset.from_list(train_rows).map(preprocess_row, remove_columns=['messages', 'metadata'])
dev_dataset = Dataset.from_list(dev_rows).map(preprocess_row, remove_columns=['messages', 'metadata'])

print(train_dataset)
print(dev_dataset)
print('sample token length:', len(train_dataset[0]['input_ids']))

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_strategy=SAVE_STRATEGY,
    eval_strategy=EVAL_STRATEGY,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

In [ ]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

metrics = trainer.evaluate()
with open(OUTPUT_DIR / 'train_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / 'eval_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('saved to', OUTPUT_DIR)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

In [ ]:
def generate_response(messages, max_new_tokens=96):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = encoded['input_ids'].shape[-1]
    return tokenizer.decode(generated[0][prompt_len:], skip_special_tokens=True)

sample_messages = dev_rows[0]['messages'][:-1]
gold_answer = dev_rows[0]['messages'][-1]['content']
pred_answer = generate_response(sample_messages)

print('USER PROMPT:')
print(sample_messages[-1]['content'][:1500])
print('\nGOLD:')
print(gold_answer)
print('\nPRED:')
print(pred_answer)

In [ ]:
%cd /kaggle/working
!zip -r qwen25-7b-fixed-span-lora.zip qwen25-7b-fixed-span-lora